In [3]:
# ================================
# STEP 1: Install & Import Libraries
# ================================
!pip install nltk scikit-learn pandas

import pandas as pd
import numpy as np
import nltk
import re
import pickle

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

# ================================
# STEP 2: Load Dataset (Upload Fake.csv & True.csv in Colab)
# ================================
fake = pd.read_csv('/content/Fake.csv', engine='python', on_bad_lines='warn')
real = pd.read_csv('/content/True.csv', engine='python', on_bad_lines='warn')

# Add labels
fake['label'] = 'FAKE'
real['label'] = 'REAL'

# Combine datasets
df = pd.concat([fake, real])

# Keep only required columns
df = df[['text', 'label']]

print("Dataset Loaded Successfully!")
print(df.head())

# ================================
# STEP 3: Text Preprocessing
# ================================
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)

print("\nText Preprocessing Done!")

# ================================
# STEP 4: Convert Labels (REAL=1, FAKE=0)
# ================================
df['label'] = df['label'].map({'REAL':1, 'FAKE':0})

# ================================
# STEP 5: Train-Test Split
# ================================
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# STEP 6: TF-IDF Vectorization
# ================================
tfidf = TfidfVectorizer(max_df=0.7)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF Applied!")

# ================================
# STEP 7: Train Model (Logistic Regression)
# ================================
model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

print("\nModel Training Completed!")

# ================================
# STEP 8: Evaluation
# ================================
y_pred = model.predict(X_test_tfidf)

print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ================================
# STEP 9: Prediction Function
# ================================
def predict_news(news):
    cleaned = clean_text(news)
    vector = tfidf.transform([cleaned])
    prediction = model.predict(vector)

    return "REAL 🟢" if prediction[0] == 1 else "FAKE 🔴"

# Test Example
print("\nSample Prediction:")
print(predict_news("Breaking: Government announces new policy"))

# ================================
# STEP 10: Save Model
# ================================
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))

print("\nModel Saved Successfully!")

# ================================
# 🎉 PROJECT COMPLETED
# ================================


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset Loaded Successfully!
                                                text label
0  Donald Trump just couldn t wish all Americans ...  FAKE
1  House Intelligence Committee Chairman Devin Nu...  FAKE
2  On Friday, it was revealed that former Milwauk...  FAKE
3  On Christmas day, Donald Trump announced that ...  FAKE
4  Pope Francis used his annual Christmas Day mes...  FAKE

Text Preprocessing Done!

TF-IDF Applied!

Model Training Completed!

Model Evaluation:
Accuracy: 0.9861915367483296

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      4733
           1       0.98      0.99      0.99      4247

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980


Confusion Matrix:
 [[4664   69]
 [  55 4192]]

Sample Prediction:
FAKE 🔴

Model Saved Successfully!
